In [ ]:
#2016-2022 county-to-county
import pandas as pd

df = pd.read_excel('/Users/judysiegel/Downloads/county-to-county-2016-2020-current-residence-sort 1.xlsx', sheet_name='Texas')

clean_df = pd.DataFrame({
    'Origin_State': df.iloc[:, 1],
    'Origin_County': df.iloc[:, 2],
    'Dest_State': df.iloc[:, 4],
    'Dest_County': df.iloc[:, 5],
    'Movers_Count': pd.to_numeric(df.iloc[:, 6], errors='coerce') # Forces numbers, drops text footnotes
})

clean_df = clean_df.dropna(subset=['Movers_Count'])

tarrant_targets = clean_df[
    (clean_df['Dest_County'].str.contains('Tarrant', na=False)) & 
    (clean_df['Origin_State'] != 'Texas') &
    (~clean_df['Origin_State'].str.contains('Total', na=False)) # Drops macro summary rows
]

top_10 = tarrant_targets.sort_values(by='Movers_Count', ascending=False).head(10)
print(top_10[['Origin_State', 'Origin_County', 'Movers_Count']])


In [ ]:
import pandas as pd

df = pd.read_excel('/Users/judysiegel/Downloads/county-to-county-2016-2020-current-residence-sort 1.xlsx', sheet_name='Texas')

for i, col_name in enumerate(df.columns):
    # Get the first valid, non-blank piece of data in this column
    sample_value = df.iloc[:, i].dropna().iloc[0] if not df.iloc[:, i].dropna().empty else "Blank"
    print(f"Index {i} ({col_name}) contains data like: {sample_value}")


In [ ]:
import pandas as pd

df = pd.read_excel('/Users/judysiegel/Downloads/county-to-county-2016-2020-current-residence-sort 1.xlsx', sheet_name='Texas')

clean_df = pd.DataFrame({
    'Dest_State': df.iloc[:, 4],       # Index 4: State of Current Residence
    'Dest_County': df.iloc[:, 5],      # Index 5: County of Current Residence
    'Origin_State': df.iloc[:, 20],    # Index 20: State of Residence 1 Year Ago
    'Origin_County': df.iloc[:, 21],   # Index 21: County of Residence 1 Year Ago
    'Movers_Count': pd.to_numeric(df.iloc[:, 36], errors='coerce') # Index 36: Real Flow Count
})

clean_df = clean_df.dropna(subset=['Movers_Count'])

tarrant_targets = clean_df[
    (clean_df['Dest_County'].str.contains('Tarrant', na=False)) & 
    (clean_df['Origin_State'] != 'Texas') &
    # Exclude macro-summary groupings if they exist in the state column
    (~clean_df['Origin_State'].str.contains('Total|United States', na=False, case=False))
]

# Extract and print the real Top 10 rows
top_10 = tarrant_targets.sort_values(by='Movers_Count', ascending=False).head(10)

print(top_10[['Origin_State', 'Origin_County', 'Movers_Count']].rename(columns={
    'Origin_State': 'Origin', 
    'Origin_County': 'County', 
    'Movers_Count': 'Total Movers'
}))

top_10.to_csv('counties-outflows.csv', index=False)
